### BulkFormer feature extraction

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

In [2]:
import math
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset,DataLoader,random_split
from torch_geometric.typing import SparseTensor

In [3]:
from utils.BulkFormer import BulkFormer

In [4]:
from model.config import model_params

In [5]:
device = 'cuda'

In [6]:
graph_path = 'data/G_gtex.pt'
weights_path = 'data/G_gtex_weight.pt'
gene_emb_path = 'data/esm2_feature_concat.pt'

In [7]:
graph = torch.load(graph_path, map_location='cpu', weights_only=False)
weights = torch.load(weights_path, map_location='cpu', weights_only=False)
graph = SparseTensor(row=graph[1], col=graph[0], value=weights).t().to(device)
gene_emb = torch.load(gene_emb_path, map_location='cpu', weights_only=False)
model_params['graph'] = graph
model_params['gene_emb'] = gene_emb

In [8]:
model = BulkFormer(**model_params).to(device)

In [9]:
ckpt_model = torch.load('model/Bulkformer_ckpt_epoch_29.pt',weights_only=False)

In [10]:
new_state_dict = OrderedDict()
for key, value in ckpt_model.items():
    new_key = key[7:] if key.startswith("module.") else key
    new_state_dict[new_key] = value

In [11]:
model.load_state_dict(new_state_dict)

<All keys matched successfully>

In [12]:
def extract_feature(expr_array, 
                    high_var_gene_idx,
                    feature_type,
                    aggregate_type,
                    device,
                    batch_size,
                    return_expr_value = False,
                    esm2_emb = None,
                    valid_gene_idx = None):

    expr_tensor = torch.tensor(expr_array,dtype=torch.float32,device=device)
    mydataset = TensorDataset(expr_tensor)
    myloader = DataLoader(mydataset, batch_size=batch_size, shuffle=False) 
    model.eval()

    all_emb_list = []
    all_expr_value_list = []


    with torch.no_grad():
        if feature_type == 'transcriptome_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)
                output, emb = model(X, [2])
                all_expr_value_list.append(output.detach().cpu().numpy())
                emb = emb[2].detach().cpu().numpy()
                emb_valid = emb[:,high_var_gene_idx,:]
     
                if aggregate_type == 'max':
                    final_emb =np.max(emb_valid, axis=1)
                elif aggregate_type == 'mean':
                    final_emb =np.mean(emb_valid, axis=1)
                elif aggregate_type == 'median':
                    final_emb =np.median(emb_valid, axis=1)
                elif aggregate_type == 'all':
                    max_emb =np.max(emb_valid, axis=1)
                    mean_emb =np.mean(emb_valid, axis=1)
                    median_emb =np.median(emb_valid, axis=1)
                    final_emb = max_emb+mean_emb+median_emb

                all_emb_list.append(final_emb)
            result_emb = np.vstack(all_emb_list)
            result_emb = torch.tensor(result_emb,device='cpu',dtype=torch.float32)

        elif feature_type == 'gene_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)
                output, emb = model(X, [2])
                emb = emb[2].detach().cpu().numpy()
                emb_valid = emb[:,valid_gene_idx,:]
                all_emb_list.append(emb_valid)
                all_expr_value_list.append(output.detach().cpu().numpy())
            all_emb = np.vstack(all_emb_list)
            all_emb_tensor = torch.tensor(all_emb,device='cpu',dtype=torch.float32)
            esm2_emb_selected = esm2_emb[valid_gene_idx]
            esm2_emb_expanded = esm2_emb_selected.unsqueeze(0).expand(all_emb_tensor.shape[0], -1, -1)  # [B, N, D]
            esm2_emb_expanded = esm2_emb_expanded.to('cpu')

            result_emb = torch.cat([all_emb_tensor, esm2_emb_expanded], dim=-1)
    
    if return_expr_value:
        return np.vstack(all_expr_value_list)
    
    else:
        return result_emb

In [13]:
def main_gene_selection(X_df, gene_list):

    to_fill_columns = list(set(gene_list) - set(X_df.columns))


    padding_df = pd.DataFrame(np.full((X_df.shape[0], len(to_fill_columns)), -10), 
                            columns=to_fill_columns, 
                            index=X_df.index)

    X_df = pd.DataFrame(np.concatenate([df.values for df in [X_df, padding_df]], axis=1), 
                        index=X_df.index, 
                        columns=list(X_df.columns) + list(padding_df.columns))
    X_df = X_df[gene_list]
    
    var = pd.DataFrame(index=X_df.columns)
    var['mask'] = [1 if i in to_fill_columns else 0 for i in list(var.index)]
    return X_df, to_fill_columns,var

In [14]:
# load demo data
# demo_df = pd.read_csv('data/demo.csv')
demo_df = pd.read_parquet("../UCLThesis/data/BIOAID_UCL_Oxford_361_combined.parquet")
demo_df = demo_df.loc[:, "5S_rRNA":]
demo_df

,5S_rRNA,A1BG,A1CF,A2M,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,AAAS,...,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11AP1,ZYG11B,ZYX,ZYXP1,ZZEF1,ZZZ3
UP2975,-9.965784,2.806248,-1.971563,4.587862,2.278299,-0.369002,0.738798,3.584693,-9.965784,3.292698,...,0.209864,-0.274774,5.047529,-0.406342,-9.965784,3.693723,8.576176,-9.965784,6.013410,3.480440
UP2978,2.705820,2.659816,-1.755116,4.881200,2.460710,-0.548148,-0.639229,3.100086,-4.415287,3.196745,...,0.629959,0.388445,4.560878,-0.902404,-9.965784,3.219514,9.128229,-9.965784,6.300789,3.459938
UP2990,2.841520,2.996421,-1.917495,6.297896,2.691242,0.435722,-1.884929,3.776965,-9.965784,3.534667,...,0.529860,0.771452,5.074128,0.055595,-9.965784,3.007938,8.685182,-9.965784,5.939457,3.545851
UP3020,3.296142,2.763925,-1.617006,3.920180,2.894636,-1.284434,-0.079042,3.714312,-9.965784,3.096036,...,-0.293795,-0.264481,4.856475,0.147134,-9.965784,2.919642,9.103588,-9.965784,6.078482,2.901291
UP3026,-9.965784,3.228935,-2.117020,7.650395,2.589678,2.328517,-0.945553,4.186653,-9.965784,3.855453,...,0.991204,1.417704,5.125280,-0.007276,-9.965784,3.626153,8.632867,-9.965784,6.339395,4.373942
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
OX.10210,1.779904,2.579028,-1.118834,3.413988,2.948571,-1.461596,0.956137,3.689575,-9.965784,3.206422,...,0.082326,0.002697,4.634716,-0.405378,-9.965784,3.753025,9.087349,-9.965784,6.036822,3.491454
OX.10245,3.846781,2.853810,-1.628879,2.828488,1.931941,-1.661713,1.243066,3.435761,-9.965784,3.005055,...,-0.357879,-0.155890,5.168501,-0.325835,-9.965784,4.456449,9.377584,-9.965784,6.238252,3.489845
OX.10318,-9.965784,2.746022,-2.737197,7.164149,1.629379,0.715447,-0.908252,3.034810,-9.965784,4.018813,...,0.975978,1.275388,4.946873,-0.491402,-9.965784,3.240878,9.041151,-9.965784,6.020115,4.426079
OX.10393,-9.965784,2.285638,-2.405857,2.969514,1.102701,-3.063042,-1.186220,2.044796,-2.150624,3.354451,...,-1.158686,-0.352694,4.550389,-0.913250,-9.965784,3.915049,9.561912,-9.965784,5.717229,2.906547


In [15]:
bulkformer_gene_info = pd.read_csv('data/bulkformer_gene_info.csv')
# fix extra row?
bulkformer_gene_info = bulkformer_gene_info[bulkformer_gene_info['ensg_id'] != '35991']

In [16]:
# bulkformer_gene_list = bulkformer_gene_info['ensg_id'].to_list()
# Use gene symbols to match with ucl data
bulkformer_gene_list = list(bulkformer_gene_info["gene_symbol"])

In [17]:

# input_df , to_fill_columns, var= main_gene_selection(X_df=demo_df,gene_list=bulkformer_gene_list)
# ucl_gene_data = pd.read_parquet("../UCLThesis/data/BIOAID_UCL_Oxford_361_combined.parquet")
input_df , to_fill_columns, var= main_gene_selection(X_df=demo_df,gene_list=bulkformer_gene_list)

In [18]:
var.reset_index(inplace=True)
valid_gene_idx = list(var[var['mask'] == 0].index)

In [19]:
high_var_gene_idx = torch.load('data/high_var_gene_list.pt',weights_only=False)
len(high_var_gene_idx)

2000

In [21]:
# Extract transcritome-level embedding
result = extract_feature(
    expr_array= input_df.values,
    high_var_gene_idx=high_var_gene_idx,
    feature_type='transcriptome_level',
    aggregate_type='max',
    device=device,
    batch_size=4,
    return_expr_value=False,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

100%|██████████| 91/91 [01:07<00:00,  1.34it/s]


In [22]:
result.shape

torch.Size([361, 640])

In [25]:
# save embeddings
ucl_embeddings = pd.DataFrame(result.numpy(), columns=[f"col_{i}" for i in range(640)])
ucl_embeddings.to_parquet("../UCLThesis/data/BIOAID_361_embeddings.parquet", index=False)

In [ ]:
# Extract gene-level embedding
result = extract_feature(
    expr_array= input_df.values[:16],
    high_var_gene_idx=high_var_gene_idx,
    feature_type='gene_level',
    aggregate_type='all',
    device=device,
    batch_size=4,
    return_expr_value=False,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:03<00:00,  1.29it/s]


In [ ]:
result.shape

torch.Size([16, 19341, 1920])

In [ ]:
result

tensor([[[-5.3849e-02, -1.1508e+00, -1.4147e-01,  ..., -9.7180e-02,
          -1.1555e-01, -6.9436e-02],
         [-8.4245e-01, -1.1047e+00, -6.0572e-02,  ..., -9.2917e-02,
          -1.0225e-02,  7.4865e-02],
         [-1.2117e+00, -3.6825e-01, -5.3327e-01,  ..., -1.5073e-01,
          -1.7446e-02,  1.4547e-01],
         ...,
         [-1.2183e+00, -1.0136e+00,  1.4344e-01,  ..., -2.8615e-02,
           1.0851e-01,  4.5788e-02],
         [-5.1785e-04, -1.4380e+00, -1.7917e-01,  ..., -6.1642e-02,
          -3.2984e-02,  1.1171e-01],
         [-1.8360e-02, -1.6205e+00, -3.0778e-01,  ...,  1.6173e-02,
          -7.6255e-02,  1.7913e-02]],

        [[-6.3123e-02, -9.4411e-01,  4.8001e-02,  ..., -9.7180e-02,
          -1.1555e-01, -6.9436e-02],
         [-9.0039e-01, -1.2653e+00,  4.9197e-02,  ..., -9.2917e-02,
          -1.0225e-02,  7.4865e-02],
         [-1.3371e+00, -1.4133e-01, -3.6086e-01,  ..., -1.5073e-01,
          -1.7446e-02,  1.4547e-01],
         ...,
         [-1.2526e+00, -1

In [ ]:
# Extract expression values
result = extract_feature(
    expr_array= input_df.values[:16],
    high_var_gene_idx=high_var_gene_idx,
    feature_type='transcriptome_level',
    aggregate_type='all',
    device=device,
    batch_size=4,
    return_expr_value=True,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

100%|██████████| 4/4 [00:03<00:00,  1.23it/s]


In [ ]:
result.shape

(16, 20010)

In [ ]:
ucl_gene_data = pd.read_parquet("../UCLThesis/data/BIOAID_UCL_Oxford_361_combined.parquet")

In [ ]:
# consider trying to match on ensembl ids instead next.
ucl_gene_list = list(ucl_gene_data.loc[:, "5S_rRNA":].columns)
print(ucl_gene_list[:4])
len(ucl_gene_list)

['5S_rRNA', 'A1BG', 'A1CF', 'A2M']


29652

In [ ]:
bulkformer_gene_list = list(bulkformer_gene_info['gene_symbol'])
len(bulkformer_gene_list)

20010

In [ ]:
ucl_gene_set = set(ucl_gene_list)
bulkformer_gene_set = set(bulkformer_gene_list)

In [ ]:
common_genes = ucl_gene_set.intersection(bulkformer_gene_set)
len(common_genes)

19338

In [ ]:
ucl_gene_set.difference(bulkformer_gene_set)

{'EIF2S2P1',
 'BECN1P2',
 'NBPF25P',
 'HMGB1P38',
 'PCGF7P',
 'RPL21P41',
 'GCSHP2',
 'PGAM1P6',
 'RPL9P2',
 'BTF3L4P1',
 'RSL24D1P1',
 'HMGB1P24',
 'SERBP1P5',
 'MIX23P3',
 'BANF1P4',
 'ATP6V0E1P4',
 'FTOP1',
 'RPL5P26',
 'SERHL',
 'NPM1P49',
 'RPLP0P5',
 'RNA5SP436',
 'H2AC3P',
 'HMGN1P17',
 'RPS15AP40',
 'GSTM2P1',
 'OR7A1P',
 'RPL21P65',
 'RAD17P2',
 'IGLV3-16',
 'FAM197Y5',
 'CHEK2P3',
 'ATP6V0CP1',
 'MTND4P3',
 'MAPK8IP1P2',
 'KRT88P',
 'RAB11FIP1P1',
 'DOC2GP',
 'CICP12',
 'PSMA1P1',
 'SLC9A7P1',
 'KRT18P23',
 'MTATP6P1',
 'RPL30P5',
 'RPL17P13',
 'NRBF2P2',
 'HMGA1P7',
 'FAF2P1',
 'RPS15AP28',
 'TRBV6-1',
 'GOLGA2P10',
 'GAPDHP34',
 'RPL21P16',
 'TRAPPC2P8',
 'FDPSP3',
 'DUXAP10',
 'ACTG1P2',
 'TRAJ5',
 'TPSP2',
 'EEF1A1P5',
 'SC4MOP',
 'MDM4P1',
 'PIGPP4',
 'RPL7AP54',
 'TPT1P2',
 'RPL7AP46',
 'RNA5SP113',
 'OFD1P4Y',
 'PES1P2',
 'OR7E35P',
 'RPL14P6',
 'RBM22P7',
 'RPL6P25',
 'MRPL42P1',
 'AQP7P3',
 'NPM1P23',
 'HMGA1P6',
 'TRAV40',
 'HSPD1P7',
 'RNA5SP292',
 'E2F6P2',
 'PTP4

In [ ]:
bulkformer_gene_set.difference(ucl_gene_set)

{'ARNTL',
 'ARNTL2',
 'BHLHB9',
 'BTBD11',
 'C10orf99',
 'C11orf53',
 'C16orf72',
 'C17orf64',
 'C19orf71',
 'C7orf61',
 'CBWD1',
 'CBWD2',
 'CBWD3',
 'CBWD5',
 'CBWD6',
 'COLCA2',
 'CYHR1',
 'DDX58',
 'ENSG00000100101',
 'ENSG00000111780',
 'ENSG00000124593',
 'ENSG00000125695',
 'ENSG00000131152',
 'ENSG00000141979',
 'ENSG00000142539',
 'ENSG00000144785',
 'ENSG00000159239',
 'ENSG00000167774',
 'ENSG00000167807',
 'ENSG00000170846',
 'ENSG00000173366',
 'ENSG00000173867',
 'ENSG00000183889',
 'ENSG00000187186',
 'ENSG00000188223',
 'ENSG00000188897',
 'ENSG00000196826',
 'ENSG00000197991',
 'ENSG00000198211',
 'ENSG00000203546',
 'ENSG00000204003',
 'ENSG00000205236',
 'ENSG00000206549',
 'ENSG00000213204',
 'ENSG00000214265',
 'ENSG00000214558',
 'ENSG00000225528',
 'ENSG00000226490',
 'ENSG00000226690',
 'ENSG00000228144',
 'ENSG00000230707',
 'ENSG00000233757',
 'ENSG00000235007',
 'ENSG00000236543',
 'ENSG00000237378',
 'ENSG00000239395',
 'ENSG00000239920',
 'ENSG00000241489',